##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Gemini Text-to-speech
<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_TTS.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API can transform text input into single speaker or multi-speaker audio podcast-like experience like in [NotebookLM](https://notebooklm.google.com/). This notebook provides an example of how to control the *Text-to-speech* (TTS) capability of the Gemini model and guide its style, accent, pace, and tone.

Before diving in the code, you should try this capability on [AI Studio](https://aistudio.google.com/prompts/new_chat?model=gemini-2.5-flash-preview-tts).

**Note that the TTS model can only do TTS, it does not have the reasoning capabilities of the Gemini models, so you can ask things like "say this in that style", but not "tell me why the sky is blue".** If that's what you want, you should use the [Live API](./Get_started_LiveAPI.ipynb) instead.

The [documentation](https://ai.google.dev/gemini-api/docs/speech-generation) is also a good place to start discovering the TTS capability.

## Setup

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

### Install and initialize the SDK

In [ ]:
!pip install -U -q "google-genai>=1.73.0"


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)

### Select a model

Audio-out is only supported by the "`tts`" models, `gemini-2.5-flash-preview-tts` and `gemini-2.5-pro-preview-tts`.

For more information about all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini) for extended information on each of them.

In [ ]:
MODEL_ID = "gemini-2.5-flash-preview-tts" # @param ["gemini-2.5-flash-preview-tts","gemini-2.5-pro-preview-tts"] {"allow-input":true, isTemplate: true}

Next create a helper function to prompt the model and play back the audio in the notebook:

In [ ]:
# @title Helper functions (just run that cell)

import contextlib
import wave
from IPython.display import Audio

file_index = 0

@contextlib.contextmanager
def wave_file(filename, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        yield wf

def play_audio_blob(blob):
  global file_index
  file_index += 1

  fname = f'audio_{file_index}.wav'
  with wave_file(fname) as wav:
    wav.writeframes(blob.data)

  return Audio(fname, autoplay=True)

def play_audio(response):
    return play_audio_blob(response.candidates[0].content.parts[0].inline_data)

## Generate a simple audio output

Let's start with something simple:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="Say 'hello, my name is Gemini!'",
  config={
    "response_modalities": ["AUDIO"],
    "speech_config": types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name="Kore"
        )
      )
    )
  },
)

The generated output is in the response `inline_data` and as you can see it's indeed audio data.

In [ ]:
blob = response.candidates[0].content.parts[0].inline_data
print(blob.mime_type)

To be able to listen to the generated audio in colab, you're going to use our helper function to write the output in a file and play it.

In [ ]:
play_audio_blob(blob)

## Choose a voice

The TTS models support 30 prebuilt voice options. You can hear all of them in [AI Studio](https://aistudio.google.com/generate-speech). Use a `SpeechConfig` with `VoiceConfig` to select a specific voice:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents="Say cheerfully: Have a wonderful day!",
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Puck',
        )
      )
    ),
  )
)

play_audio(response)

## Control how the model speaks

The Gemini TTS model differentiates itself from traditional TTS by using a large language model that knows not only **what** to say, but also **how** to say it.

You can control style, tone, accent, and pace using natural language in your prompts. For best results, the official documentation recommends structuring your prompt with the following components:

### Prompting Strategy: Audio Profile, Scene, and Director's Notes

A robust prompt ideally includes these elements:

- **Audio Profile** — Establishes a persona for the voice: character identity, name, archetype, and background.
- **Scene** — Sets the stage, describing the physical environment and the emotional "vibe".
- **Director's Notes** — Performance guidance covering style, pacing, accent, and articulation.
- **Sample Context** — Gives the model a contextual starting point so your "virtual actor" enters the scene naturally.
- **Transcript** — The actual text the model will speak.

Here's a full example:

In [ ]:
prompt = """
# AUDIO PROFILE: Jaz R.
## "The Morning Hype"

## THE SCENE: The London Studio
It is 10:00 PM in a glass-walled studio overlooking the moonlit London skyline,
but inside, it is blindingly bright. The red "ON AIR" tally light is blazing.
Jaz is standing up, not sitting, bouncing on the balls of their heels to the
rhythm of a thumping backing track.

### DIRECTOR'S NOTES

Style:
* The "Vocal Smile": You must hear the grin in the audio. The soft palate is
  always raised to keep the tone bright, sunny, and explicitly inviting.
* Dynamics: High projection without shouting. Punchy consonants and elongated
  vowels on excitement words (e.g., "Beauuutiful morning").

Pace:
Speaks at an energetic pace, keeping up with the fast music.
A "bouncing" cadence. High-speed delivery with fluid transitions.

Accent:
Jaz is from Brixton, London

### SAMPLE CONTEXT
Jaz is the industry standard for Top 40 radio, high-octane event promos, or
any script that requires a charismatic Estuary accent and infectious energy.

#### TRANSCRIPT
Yes, massive vibes in the studio! You are locked in and it is absolutely
popping off in London right now. If you're stuck on the tube, or just sat
there pretending to work... stop it. Seriously, I see you. Turn this up!
We've got the project roadmap landing in three, two... let's go!
"""

response = client.models.generate_content(
  model=MODEL_ID,
  contents=prompt,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Kore',
        )
      )
    ),
  )
)

play_audio(response)

### Tips for Director's Notes

You don't need to include every element — sometimes giving the model space to fill in the gaps helps naturalness (just like a talented actor).

**Style** — Sets the tone of the generated speech. Be descriptive: *"Infectious enthusiasm. The listener should feel like they are part of a massive, exciting community event."* works better than simply saying *"energetic and enthusiastic"*. You can even try voiceover industry terms like "vocal smile".

**Accent** — The more specific you are, the better. Use *"British English accent as heard in Croydon, England"* rather than *"British accent"*.

**Pacing** — Control overall speed and variation. Examples range from simple (*"Speak as fast as possible"*) to complex (*"The Drift: The tempo is incredibly slow and liquid. Words bleed into each other. There is zero urgency."*).

### Simple style control

You can also control style with much simpler prompts. Here are a few examples:

In [ ]:
response = client.models.generate_content(
  model=MODEL_ID,
  contents='Say in a spooky whisper: "By the pricking of my thumbs... Something wicked this way comes"',
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      voice_config=types.VoiceConfig(
        prebuilt_voice_config=types.PrebuiltVoiceConfig(
          voice_name='Enceladus',
        )
      )
    ),
  )
)

play_audio(response)

Try using a voice option that corresponds to the style or emotion you want to convey, to emphasize it even more. For example, Enceladus's breathiness might emphasize "tired" and "bored", while Puck's upbeat tone could complement "excited" and "happy".

## Multi-speaker TTS

For multi-speaker audio, use a `MultiSpeakerVoiceConfig` with each speaker (up to 2) configured as a `SpeakerVoiceConfig`. The speaker names in the config **must match** the names used in your prompt:

In [ ]:
prompt = """TTS the following conversation between Joe and Jane:
Joe: How's it going today Jane?
Jane: Not too bad, how about you?"""

response = client.models.generate_content(
  model=MODEL_ID,
  contents=prompt,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
        speaker_voice_configs=[
          types.SpeakerVoiceConfig(
            speaker='Joe',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Kore',
              )
            )
          ),
          types.SpeakerVoiceConfig(
            speaker='Jane',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Puck',
              )
            )
          ),
        ]
      )
    )
  )
)

play_audio(response)

You can also provide individual style guidance for each speaker in a multi-speaker prompt:

In [ ]:
prompt = """Make Speaker1 sound tired and bored, and Speaker2 sound excited and happy:
Speaker1: So... what's on the agenda today?
Speaker2: You're never going to guess!"""

response = client.models.generate_content(
  model=MODEL_ID,
  contents=prompt,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
        speaker_voice_configs=[
          types.SpeakerVoiceConfig(
            speaker='Speaker1',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Enceladus',
              )
            )
          ),
          types.SpeakerVoiceConfig(
            speaker='Speaker2',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Puck',
              )
            )
          ),
        ]
      )
    )
  )
)

play_audio(response)

## Generate a transcript, then convert to audio

The TTS models only output audio, but you can use a regular Gemini model to generate a transcript first, then pass it to the TTS model. This is useful for creating podcast-style content:

In [ ]:
# Step 1: Generate a transcript with a regular Gemini model
transcript = client.models.generate_content(
  model="gemini-2.5-flash",
  contents="""Generate a short transcript around 100 words that reads like
  it was clipped from a podcast by excited herpetologists.
  The hosts names are Dr. Anya and Liam."""
).text

print("Generated transcript:")
print(transcript)

In [ ]:
# Step 2: Convert the transcript to audio with the TTS model
response = client.models.generate_content(
  model=MODEL_ID,
  contents=transcript,
  config=types.GenerateContentConfig(
    response_modalities=["AUDIO"],
    speech_config=types.SpeechConfig(
      multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
        speaker_voice_configs=[
          types.SpeakerVoiceConfig(
            speaker='Dr. Anya',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Kore',
              )
            )
          ),
          types.SpeakerVoiceConfig(
            speaker='Liam',
            voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                voice_name='Puck',
              )
            )
          ),
        ]
      )
    )
  )
)

play_audio(response)

## Further reading

- [Speech generation documentation](https://ai.google.dev/gemini-api/docs/speech-generation)
- [Voice Library in AI Studio](https://aistudio.google.com/apps/bundled/voice-library?showPreview=true)
- [Live API](https://ai.google.dev/gemini-api/docs/live) for interactive, real-time audio generation
- [Audio understanding](https://ai.google.dev/gemini-api/docs/audio) for working with audio inputs